[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpinpa3/datos-maritimos/blob/main/02-era5-historico-del-mar/era5_dias_mala_mar.ipynb)

# Días de mala mar en un punto con ERA5

Este notebook descarga de Copernicus la serie horaria de **altura significativa de ola** (oleaje de viento más mar de fondo) en un punto de mar abierto y responde a una pregunta sencilla: **¿cuántos días al año se superan 2, 3 y 4 metros de ola, y en qué meses?**

Punto por defecto: 37,5° N, 0,5° W, en mar abierto frente a la costa de Cartagena. Puede cambiarse en la celda de parámetros.

**Fuente:** Copernicus Climate Change Service (C3S) / ECMWF, ERA5. Contiene información modificada del Copernicus Climate Change Service.

## 1. Requisitos

1. Crear una cuenta gratuita en el [Climate Data Store](https://cds.climate.copernicus.eu).
2. Aceptar la licencia del conjunto de datos ERA5 (pestaña *Download* del conjunto, al final de la página).
3. Copiar la clave personal de la API desde el perfil de usuario.

La clave se pide al ejecutar la celda de configuración y no se guarda en el notebook.

In [ ]:
!pip install -q "cdsapi>=0.7.4" pandas matplotlib

In [ ]:
import os
from getpass import getpass

clave = getpass("Clave de la API del Climate Data Store: ")
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {clave}\n")
print("Configuración guardada")

## 2. Parámetros

- `LATITUD` y `LONGITUD`: el punto que se quiere estudiar. ERA5 devuelve el nodo de malla de oleaje más cercano (malla de 0,5°).
- `INICIO` y `FIN`: periodo de análisis. Treinta años es el periodo habitual para hablar de «clima» de una zona.
- `UMBRALES_M`: alturas de ola, en metros, cuya superación se cuenta.

In [ ]:
LATITUD = 37.5     # grados norte
LONGITUD = -0.5    # grados este (negativo = oeste)
INICIO = "1995-01-01"
FIN = "2024-12-31"
UMBRALES_M = [2, 3, 4]
FICHERO = "era5_oleaje_punto.csv"

## 3. Descarga

Se usa el conjunto *ERA5 hourly time-series data on single levels*, pensado para series largas en un solo punto. La petición entra en una cola del servidor de Copernicus y puede tardar unos minutos.

In [ ]:
import cdsapi

peticion = {
    "variable": ["significant_height_of_combined_wind_waves_and_swell"],
    "location": {"longitude": LONGITUD, "latitude": LATITUD},
    "date": [f"{INICIO}/{FIN}"],
    "data_format": "csv",
}
cdsapi.Client().retrieve("reanalysis-era5-single-levels-timeseries", peticion).download(FICHERO)
print("Descargado:", FICHERO)

## 4. Lectura y preparación

El fichero puede venir directamente en CSV o comprimido en ZIP según la versión del servicio; la celda contempla ambos casos y localiza las columnas de tiempo y de altura de ola por su nombre.

In [ ]:
import zipfile
import pandas as pd

def leer_serie(ruta):
    if zipfile.is_zipfile(ruta):
        with zipfile.ZipFile(ruta) as z:
            nombre = [n for n in z.namelist() if n.endswith(".csv")][0]
            with z.open(nombre) as f:
                return pd.read_csv(f)
    return pd.read_csv(ruta)

df = leer_serie(FICHERO)
col_tiempo = next(c for c in df.columns if "time" in c.lower())
col_ola = next(c for c in df.columns if c.lower() in ("swh", "significant_height_of_combined_wind_waves_and_swell"))

serie = (df[[col_tiempo, col_ola]]
         .rename(columns={col_tiempo: "tiempo", col_ola: "hs_m"})
         .assign(tiempo=lambda d: pd.to_datetime(d["tiempo"]))
         .dropna()
         .set_index("tiempo")
         .sort_index())

print(f"{len(serie):,} horas, de {serie.index.min():%d/%m/%Y} a {serie.index.max():%d/%m/%Y}")
print(f"Nodo devuelto: {df.filter(like='lat').iloc[0, 0] if not df.filter(like='lat').empty else 'no indicado'}, "
      f"{df.filter(like='lon').iloc[0, 0] if not df.filter(like='lon').empty else 'no indicado'}")
serie.describe().round(2)

## 5. Días al año por encima de cada umbral

Un día cuenta como «día de mala mar» para un umbral si la **altura significativa máxima** de ese día lo supera en alguna hora.

In [ ]:
import matplotlib.pyplot as plt

maximo_diario = serie["hs_m"].resample("D").max().dropna()

dias_por_anio = pd.DataFrame({
    f"> {u} m": (maximo_diario > u).groupby(maximo_diario.index.year).sum()
    for u in UMBRALES_M
})
dias_por_anio.index.name = "año"

ax = dias_por_anio.plot(kind="bar", figsize=(12, 4.5), width=0.8)
ax.set_title(f"Días al año con altura significativa de ola por encima del umbral\n"
             f"ERA5 · {LATITUD}° N, {abs(LONGITUD)}° {'W' if LONGITUD < 0 else 'E'}")
ax.set_xlabel("")
ax.set_ylabel("Días")
ax.legend(title="Umbral", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.figtext(0.01, -0.02, "Fuente: Copernicus Climate Change Service (C3S) / ECMWF, ERA5", fontsize=8, color="gray")
plt.tight_layout()
plt.savefig("dias_mala_mar_por_anio.png", dpi=150, bbox_inches="tight")
plt.show()

resumen = dias_por_anio.agg(["mean", "min", "max"]).T.round(1)
resumen.columns = ["media anual", "mínimo", "máximo"]
resumen

## 6. Estacionalidad: porcentaje de días por mes

Qué parte de los días de cada mes, en el conjunto del periodo, supera cada umbral. Es la información útil para planificar en qué época del año se concentra el riesgo.

In [ ]:
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
pct_mes = pd.DataFrame({
    f"> {u} m": (maximo_diario > u).groupby(maximo_diario.index.month).mean() * 100
    for u in UMBRALES_M
})
pct_mes.index = meses

ax = pct_mes.plot(kind="bar", figsize=(10, 4.5), width=0.8)
ax.set_title("Porcentaje de días del mes con altura significativa de ola por encima del umbral")
ax.set_ylabel("% de días")
ax.set_xlabel("")
ax.legend(title="Umbral", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0)
plt.figtext(0.01, -0.02, "Fuente: Copernicus Climate Change Service (C3S) / ECMWF, ERA5", fontsize=8, color="gray")
plt.tight_layout()
plt.savefig("estacionalidad_mala_mar.png", dpi=150, bbox_inches="tight")
plt.show()

pct_mes.round(1)

## 7. Cómo interpretar los resultados

- **Altura significativa no es ola máxima.** La altura significativa es, aproximadamente, la media del tercio de olas más altas. Las olas individuales máximas pueden ser bastante mayores.
- **Es un punto de mar abierto.** La malla de oleaje es de 0,5°: el resultado no representa el interior de un puerto ni zonas costeras abrigadas.
- **El reanálisis suaviza extremos.** Los temporales muy locales y los picos pueden quedar por debajo de lo medido por una boya cercana.
- **Uso:** sirve para comparar zonas, épocas del año y tendencias. Para decisiones operativas concretas debe complementarse con datos de boyas y previsiones oficiales.

**Fuente:** Copernicus Climate Change Service (C3S) / ECMWF (2023-hoy), *ERA5 hourly time-series data on single levels from 1940 to present*. Contiene información modificada del Copernicus Climate Change Service.